# Matching And Querying Walkthrough

This notebook shows how `graph_to_vec` handles candidate matching: noisy records become candidate pairs, candidate pairs become evidence graphs, and a graph classifier ranks likely matches.

## 1. Setup

Use the local `.venv` kernel. No container is needed.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "graph_to_vec").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from examples.matching_workflow import (
    RANDOM_STATE,
    build_candidate_embedding_index,
    build_labeled_candidate_pairs,
    make_record_table,
    query_for_record,
    train_match_ranker,
)
from graph_to_vec import MatchGraphBuilder

## 2. Noisy Records

The table below simulates records from CRM, support, and billing systems. `entity_id` is the training truth used to label known matches.

In [ ]:
records = make_record_table()
records[["id", "entity_id", "name", "email_domain", "company", "zip"]]

## 3. Candidate Generation

`CandidatePairBuilder` blocks on email domain, computes field similarities, and labels pairs whose records share the same `entity_id`.

In [ ]:
pairs = build_labeled_candidate_pairs(records)
summary = pairs["label"].value_counts().rename(index={0: "not_match", 1: "match"})
display(summary)

pairs[
    ["left_id", "right_id", "label", "candidate_score", "sim_name", "sim_company"]
].sort_values("candidate_score", ascending=False).head(12)

## 4. Evidence Graphs

Every candidate pair becomes a small heterogeneous graph: a candidate node, left/right record nodes, and field-level evidence nodes.

In [ ]:
graphs = MatchGraphBuilder().fit_transform(pairs.head(3), left_records=records)
graph = graphs[0]

print(graph.graph)
print("nodes", graph.number_of_nodes())
print("edges", graph.number_of_edges())
pd.DataFrame(
    [
        {"node": node, **attrs}
        for node, attrs in graph.nodes(data=True)
    ]
).head(10)

## 5. Train A Match Ranker

`MatchRanker` trains a graph classification pipeline over the evidence graphs and exposes ranking/query helpers.

In [ ]:
train_pairs, test_pairs = train_test_split(
    pairs,
    test_size=0.35,
    random_state=RANDOM_STATE,
    stratify=pairs["label"],
)

ranker, _, _ = train_match_ranker(records, pairs)
predictions = ranker.predict(records, pairs=test_pairs)
report = classification_report(
    test_pairs["label"],
    predictions,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).T.round(3)

## 6. Rank And Query Matches

`rank` sorts candidate pairs globally. `query` filters the ranking to one record id.

In [ ]:
ranked = ranker.rank(records, pairs=pairs)
ranked[
    ["rank", "left_id", "right_id", "label", "prediction", "match_score"]
].head(12)

In [ ]:
query_for_record(ranker, records, "crm:001")

## 7. Embedding Query

The same candidate evidence graphs can be embedded and searched with `EmbeddingIndex`.

In [ ]:
build_candidate_embedding_index(ranker, records, pairs)